# Chapter 05. 분석을 믿을 수 있게 만드는 데이터 전처리

## 0. 제출 정보

- 이름: 이성민
- GitHub ID: minilee0307-dot
- 작성일·분석 고정 기준일: 2026-10-08
- 최종 제출 URL(게시·제출 대상): https://github.com/minilee0307-dot/llm-data-analysis-study/blob/main/chapter05/chapter05.ipynb
- 실습 실행 상태: **COMPLETE** — 코드 21셀을 실행하고 출력을 저장했다.
- 전체 제출 상태: **PARTIAL** — GitHub 게시·표시 확인과 eTL 접수가 남아 있다.
- GitHub 게시·렌더링 확인 / eTL 제출: 확인 대기. 로컬 실행 완료와 구분한다.

공식 `notebooks/ch05_data_preprocessing.ipynb`를 바탕으로 실행 흐름, 추가 실습 6문제와 답안 템플릿을 통합했다. 참조한 공식 커밋은 `b1597e10c87f5c461bdb7e3f8f9b8454ba8cbedc`이다.

공식 스크립트가 지정한 `practice/chapter05/data/raw`의 **전처리 전 전용 샘플**을 제출 저장소 `data/raw`에 복사했다. 공통 정상 샘플을 다시 생성한 것이 아니다. [원본 출처·해시](../data/raw/README.md), [전처리 코드](../src/preprocessing.py), [재실행 스크립트](../scripts/preprocess_data.py)를 함께 제출한다.

작성과 실행에는 Codex의 지원을 받았고 LLM 제안과 실제 검증 결과를 구분했다. 기존 수업 가상환경을 재사용했으며 실제 Python 경로를 아래에 출력한다. 제출 저장소에서는 루트 `requirements.txt`로 환경을 재현할 수 있다.

핵심 Evidence 1–6은 실제 실행된 셀 출력이다.

공식 [실습 가이드](https://github.com/GilbertMoon/llm-data-analysis-course/blob/b1597e10c87f5c461bdb7e3f8f9b8454ba8cbedc/practice/chapter05/chapter05.md)와 [답안 양식](https://github.com/GilbertMoon/llm-data-analysis-course/blob/b1597e10c87f5c461bdb7e3f8f9b8454ba8cbedc/practice/chapter05/templates/chapter05_assignment.md)의 항목도 확인했다. eTL 상세 안내도 직접 확인했다. Chapter 05 전용 dirty CSV 4개를 원본 상태로 사용하고, 실행 결과와 Markdown 답안을 저장한 개인 GitHub Notebook 파일 URL을 제출하는 조건에 맞췄다.


온라인 확인 기록(2026-10-08): eTL 상세 안내에 접근해 과제 요구사항과 마감을 확인했다. GitHub 게시·표시와 eTL 접수는 성공 여부를 따로 확인한다.


## 실행 환경과 원본 보존

원본을 바꾸기 전에 네 파일의 존재·크기·SHA-256을 기록한다. 현재 위치가 저장소 루트 또는 `chapter05`여도 입력 경로를 찾도록 했다.


In [1]:
from pathlib import Path
import sys, hashlib, subprocess, json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'data/raw/customers.csv').exists() and (p / 'src/preprocessing.py').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocessing import run_preprocessing, load_raw_data, quality_summary, validate_relationships, sha256

RAW_DIR = PROJECT_ROOT / 'data/raw'
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
print('실제 Python:', sys.executable)
print('Python:', sys.version.split()[0], '/ pandas:', pd.__version__, '/ NumPy:', np.__version__)
print('프로젝트 루트:', PROJECT_ROOT)
print('입력:', RAW_DIR, '\n출력:', PROCESSED_DIR, '\n보고서:', REPORT_DIR)
raw_paths = [RAW_DIR / f'{name}.csv' for name in ['customers','products','orders','order_items']]
assert all(p.exists() and p.stat().st_size > 0 for p in raw_paths)
raw_hash_start = {p.name: sha256(p) for p in raw_paths}
display(pd.DataFrame([{'file':p.name, 'exists':p.exists(), 'bytes':p.stat().st_size, 'sha256':sha256(p)} for p in raw_paths]))
print('원본 CSV 4개 확인: PASS')


실제 Python: /Users/sungmiiinn/Desktop/LLM을 활용한 빅데이터 분석/llm-data-analysis-course/.venv/bin/python
Python: 3.14.3 / pandas: 3.0.5 / NumPy: 2.5.3
프로젝트 루트: /Users/sungmiiinn/.codex/.chatgpt-projects/g-p-6aa22dd39878819190a2145e7a2652e2/assignment
입력: /Users/sungmiiinn/.codex/.chatgpt-projects/g-p-6aa22dd39878819190a2145e7a2652e2/assignment/data/raw 
출력: /Users/sungmiiinn/.codex/.chatgpt-projects/g-p-6aa22dd39878819190a2145e7a2652e2/assignment/data/processed 
보고서: /Users/sungmiiinn/.codex/.chatgpt-projects/g-p-6aa22dd39878819190a2145e7a2652e2/assignment/reports


,file,exists,bytes,sha256
0,customers.csv,True,5846,ef4e554eaae1bb0cb7605aa9e4503aad1e68bbf1eae909...
1,products.csv,True,3906,cad0a825a29078d9451aa74fdfa9e61d474f442d2e3778...
2,orders.csv,True,11345,bb32cb334b9e4920f68e6ef55df17d414bdc51f7c28a58...
3,order_items.csv,True,14559,e59b236328b6b094fdcbc0142af35b7ccb54e4246e1a57...


원본 CSV 4개 확인: PASS


## 1. 전처리 전 상태 기록 — Evidence 1

행 수뿐 아니라 실제 컬럼·dtype·결측·전체 행 중복·고유 ID 중복과 FK를 확인한다. 문자열 숫자, 날짜와 범주 표기 차이가 이번 전처리의 주요 대상이다.


In [2]:
raw_data = load_raw_data(PROJECT_ROOT)
raw_copy = {name: df.copy(deep=True) for name, df in raw_data.items()}
display(quality_summary(raw_data))
display(pd.DataFrame([{'dataset':name,'column':col,'dtype':str(dtype)}
                      for name,df in raw_data.items() for col,dtype in df.dtypes.items()]))
missing_all = pd.concat([pd.DataFrame({'dataset':name,'column':df.columns,
    'missing_count':df.isna().sum().values, 'missing_ratio_pct':(df.isna().mean()*100).round(2).values})
    for name,df in raw_data.items()], ignore_index=True)
display(missing_all)
display(validate_relationships(raw_data))


,dataset,rows,columns,missing_cells,full_row_duplicates,key_missing,key_duplicates
0,customers,152,6,6,2,0,2
1,products,102,4,0,2,0,2
2,orders,302,5,0,2,0,2
3,order_items,765,5,0,2,0,2


,dataset,column,dtype
0,customers,customer_id,int64
1,customers,name,str
2,customers,gender,str
3,customers,age,str
4,customers,city,str
5,customers,signup_date,str
6,products,product_id,int64
7,products,product_name,str
8,products,category,str
9,products,price,str


,dataset,column,missing_count,missing_ratio_pct
0,customers,customer_id,0,0.00
1,customers,name,0,0.00
2,customers,gender,0,0.00
3,customers,age,4,2.63
4,customers,city,2,1.32
5,customers,signup_date,0,0.00
6,products,product_id,0,0.00
7,products,product_name,0,0.00
8,products,category,0,0.00
9,products,price,0,0.00


,relationship,invalid_count
0,orders.customer_id → customers.customer_id,2
1,order_items.order_id → orders.order_id,2
2,order_items.product_id → products.product_id,2


### 결과 관찰

원본은 customers **152×6**, products **102×4**, orders **302×5**, order_items **765×5**이다. 네 파일 모두 전체 중복과 고유 ID 중복이 각각 2건이다. 원래 결측은 age 4셀, city 2셀이고 다른 원래 컬럼은 0셀이다. FK 미연결 행은 세 관계에서 각각 2건씩 확인했다. 문자열 컬럼은 pandas 3에서 `str`로 읽혔다.

### 나의 해석과 판단

결측이 0이어도 안전한 데이터라고 판단할 수 없다. `unknown`과 잘못된 날짜는 문자열이라 결측으로 집계되지 않기 때문이다. 먼저 표기를 정리하고 변환 실패를 따로 센 뒤 중복·관계를 검증하기로 했다. 전체 행이 같을 때만 중복을 제거하고, 같은 ID에 다른 내용이 있으면 임의로 하나를 고르지 않는다.

### 업무·분석적 의미

변환 실패를 놓치면 주문 금액이나 월별 건수가 달라질 수 있다. 네 원본 파일을 모두 기록해 일부 파일만 정상이라는 이유로 전체 준비를 완료했다고 판단하지 않았다.

### 한계와 추가 확인 사항

이 데이터는 문제를 의도적으로 포함한 합성 샘플이다. 실제 고객 오류나 회사 데이터 품질의 비율로 일반화할 수 없다.


## 2. 문자열·타입·결측 처리 — Evidence 2

전처리는 `run_preprocessing(PROJECT_ROOT)`로 복사본에서 수행한다. Notebook과 스크립트에서 같은 함수를 사용하고, 원래 행 번호로 변경 내역을 추적한다. 아래 함수 실행이 전체 처리의 실제 실행이며 이후 셀은 그 결과를 단계별로 검증한다.


In [3]:
result = run_preprocessing(PROJECT_ROOT)
clean_data = result['clean']
customers_clean, products_clean = clean_data['customers'], clean_data['products']
orders_clean, order_items_clean = clean_data['orders'], clean_data['order_items']
for name in raw_data:
    pd.testing.assert_frame_equal(raw_data[name], raw_copy[name])
assert raw_hash_start == result['manifest']['raw_sha256_after']
print('원본 DataFrame 및 원본 파일 보존: PASS')
display(result['string_checks'])
display(result['conversions'])


원본 DataFrame 및 원본 파일 보존: PASS


,dataset,column,trimmed_cells,blank_to_missing
0,customers,name,0,0
1,customers,gender,0,0
2,customers,age,0,0
3,customers,city,5,0
4,customers,signup_date,0,0
5,products,product_name,0,0
6,products,category,3,0
7,products,price,0,0
8,orders,order_date,0,0
9,orders,payment_method,5,0


,dataset,column,kind,input_rows,original_missing,parse_failures_before_formatting,comma_removed,recovered_unit_suffix,new_parse_failures,missing_after_conversion
0,customers,age,numeric,150,4,2,0,0,2,6
1,products,price,numeric,100,0,6,4,0,2,2
2,order_items,quantity,numeric,763,0,5,0,3,2,2
3,order_items,unit_price,numeric,763,0,7,5,0,2,2
4,customers,signup_date,date,150,0,2,0,0,2,2
5,orders,order_date,date,300,0,3,0,0,3,3


In [4]:
# 쉼표/단위 정리로 회복된 원문과 남은 변환 실패를 따로 확인한다.
numeric_audit = result['audit'].query("action == 'numeric_conversion'")
recoverable = numeric_audit[numeric_audit['before'].str.contains(r',|개', regex=True, na=False)]
failed = numeric_audit[numeric_audit['after'].eq('<NA>') | numeric_audit['after'].eq('nan')]
print('숫자 형식 정리 사례')
display(recoverable)
print('형식 정리 후에도 남은 숫자 변환 실패')
display(failed)
print('최종 dtype')
display(pd.DataFrame([{'dataset':name,'column':col,'dtype':str(dtype)}
                      for name,df in clean_data.items() for col,dtype in df.dtypes.items()]))


숫자 형식 정리 사례


,dataset,raw_row_number,column,action,before,after
25,products,10,price,numeric_conversion,"75,000",75000
27,products,23,price,numeric_conversion,"181,000",181000
28,products,62,price,numeric_conversion,"28,000",28000
30,products,88,price,numeric_conversion,"93,000",93000
31,order_items,18,quantity,numeric_conversion,3개,3
32,order_items,92,quantity,numeric_conversion,2개,2
33,order_items,206,quantity,numeric_conversion,5개,5
36,order_items,45,unit_price,numeric_conversion,"52,000",52000
37,order_items,189,unit_price,numeric_conversion,"17,000",17000
39,order_items,367,unit_price,numeric_conversion,"161,000",161000


형식 정리 후에도 남은 숫자 변환 실패


,dataset,raw_row_number,column,action,before,after
13,customers,25,age,numeric_conversion,unknown,<NA>
15,customers,98,age,numeric_conversion,unknown,<NA>
26,products,19,price,numeric_conversion,unknown,<NA>
29,products,74,price,numeric_conversion,unknown,<NA>
34,order_items,320,quantity,numeric_conversion,three,<NA>
35,order_items,448,quantity,numeric_conversion,unknown,<NA>
38,order_items,278,unit_price,numeric_conversion,unknown,<NA>
40,order_items,434,unit_price,numeric_conversion,not-a-number,<NA>


최종 dtype


,dataset,column,dtype
0,customers,customer_id,int64
1,customers,name,string
2,customers,gender,string
3,customers,age,Float64
4,customers,city,string
5,customers,signup_date,datetime64[us]
6,customers,age_imputed,bool
7,customers,age_outlier_candidate,boolean
8,customers,city_imputed,bool
9,customers,signup_date_invalid,bool


In [5]:
# 대체값의 영향도 확인한다. 150은 이상 후보로 보존하고 대체용 중앙값에서는 제외했다.
age_before = pd.to_numeric(raw_data['customers'].drop_duplicates()['age'], errors='coerce')
age_before_valid = age_before[age_before.between(18,100)]
age_after_valid = customers_clean.loc[~customers_clean['age_outlier_candidate'],'age']
display(pd.DataFrame([
    {'range':'관측된 18–100세(대체 전)','count':len(age_before_valid),'mean':age_before_valid.mean(),'median':age_before_valid.median()},
    {'range':'대체 후(이상 후보 제외)','count':len(age_after_valid),'mean':age_after_valid.mean(),'median':age_after_valid.median()},
]))
print('나이 대체 중앙값:', result['manifest']['age_imputation_median'])
print('나이 대체:', int(customers_clean['age_imputed'].sum()), '도시 Unknown 대체:', int(customers_clean['city_imputed'].sum()))
display(customers_clean[['customer_id','age','age_imputed','age_outlier_candidate','city_imputed']].query('age_imputed or age_outlier_candidate or city_imputed'))


,range,count,mean,median
0,관측된 18–100세(대체 전),143,43.293706,41.0
1,대체 후(이상 후보 제외),149,43.201342,41.0


나이 대체 중앙값: 41.0
나이 대체: 6 도시 Unknown 대체: 2


,customer_id,age,age_imputed,age_outlier_candidate,city_imputed
11,12,41.0,True,False,False
23,24,41.0,True,False,False
32,33,150.0,False,True,False
40,41,41.0,False,False,True
52,53,41.0,True,False,False
87,88,41.0,True,False,False
88,89,68.0,False,False,True
96,97,41.0,True,False,False
120,121,41.0,True,False,False


In [6]:
# 날짜 실패를 원래 결측과 구분하고, NaT와 미래 후보를 보존했는지 확인한다.
for name,df,col in [('customers',customers_clean,'signup_date'),('orders',orders_clean,'order_date')]:
    print(name, col, '실패/결측:',int(df[col].isna().sum()), '최소:',df[col].min(), '최대:',df[col].max())
    display(df.loc[df[f'{col}_invalid'] | df[f'{col}_future_candidate'],
                   [df.columns[0],col,f'{col}_invalid',f'{col}_future_candidate']])
print('order_month/date/요일 결측을 날짜 오류 9건으로 중복 계산하지 않는다.')


customers signup_date 실패/결측: 2 최소: 2023-07-21 00:00:00 최대: 2026-08-24 00:00:00


,customer_id,signup_date,signup_date_invalid,signup_date_future_candidate
64,65,NaT,True,False
108,109,NaT,True,False


orders order_date 실패/결측: 3 최소: 2025-07-01 00:00:00 최대: 2028-12-31 00:00:00


,order_id,order_date,order_date_invalid,order_date_future_candidate
71,72,NaT,True,False
141,143,NaT,True,False
212,214,NaT,True,False
254,256,2027-01-15,False,True
284,287,2028-12-31,False,True


order_month/date/요일 결측을 날짜 오류 9건으로 중복 계산하지 않는다.


### 결과 관찰

공백 정리 대상은 city 5셀, category 3셀, payment_method 5셀이다. 빈 문자열을 새 결측으로 바꾼 건수는 0이다. 완전중복을 제거한 변환 대상은 150/100/300/763행이다.

수량의 `3개`, `2개`, `5개` 3건은 단위만 제거해 복구했다. 정리 후 새 숫자 변환 실패는 age 2건, price 2건, quantity 2건, unit_price 2건이다. age는 원래 결측 4건과 실패 2건을 합친 6건을 **41세**로 대체했다. city 결측 2건은 Unknown으로 구분했다. 가입일은 2건, 주문일은 3건이 변환되지 않았다. 기준일 2026-10-08보다 뒤인 주문일 후보는 **2건**이다.

### 나의 해석과 판단

숫자+`개`는 단위를 제거해도 수량 의미가 명확하지만 `three`, `unknown`은 이 실습에서 추측해 복구하지 않았다. 쉼표 정리 전 실패와 정리 후 실패를 구분했다. 나이는 관측된 18–100세의 중앙값을 쓰고 대체 플래그를 남겼다. 나이 150은 대체용 중앙값 계산에서 제외하되 원래 값을 후보로 보존했다. ID·거래 금액·날짜는 대표값으로 채우지 않았다.

### 업무·분석적 의미

대체한 41은 실제 나이가 아니며 연령 분포를 중앙으로 모을 수 있다. 대체 여부를 남기면 다음 분석에서 포함/제외 결과를 비교할 수 있다. 날짜 실패를 NaT로 남기면 임의 날짜를 넣어 월별 집계를 오염시키는 일을 피할 수 있다.

### 한계와 추가 확인 사항

변환 실패한 원래 값은 원본과 변경 기록에 남아 있다. 실제 의미를 원본 시스템에서 확인하지는 못했다. 미래 날짜를 실제 오류라고 확정하지 않았으며 다음 장에서 과거 거래 분석 범위를 별도로 제한한다.


## 3. 중복·범주 표준화·이상값 후보 — Evidence 3


In [7]:
display(result['before'][['dataset','full_row_duplicates','key_missing','key_duplicates']])
display(result['after'][['dataset','full_row_duplicates','key_missing','key_duplicates']])
print('완전중복 격리 행 수')
display(result['exclusions'].query("stage == 'duplicate'").groupby('dataset').size().reset_index(name='rows'))
print('주문 상세의 order_id 반복 수:', int(order_items_clean['order_id'].duplicated().sum()), '(FK의 정상 반복, 삭제 대상 아님)')
print('충돌 PK 격리:', int(result['exclusions']['reasons'].str.contains('conflicting_primary_key').sum()))


,dataset,full_row_duplicates,key_missing,key_duplicates
0,customers,2,0,2
1,products,2,0,2
2,orders,2,0,2
3,order_items,2,0,2


,dataset,full_row_duplicates,key_missing,key_duplicates
0,customers,0,0,0
1,products,0,0,0
2,orders,0,0,0
3,order_items,0,0,0


완전중복 격리 행 수


,dataset,rows
0,customers,2
1,order_items,2
2,orders,2
3,products,2


주문 상세의 order_id 반복 수: 427 (FK의 정상 반복, 삭제 대상 아님)
충돌 PK 격리: 0


In [8]:
print('상태 표준화 전: 완전중복 제거 후, FK 격리 전')
display(result['status_before'])
print('최종 clean 상태 빈도')
display(result['status_after'])
print('실제로 바뀐 표기')
display(result['audit'].query("action in ['status_standardization','category_exact_mapping']"))
print('허용값 밖 상태:', int(orders_clean['order_status_unexpected'].sum()))
assert orders_clean['order_status'].isin(['completed','cancelled','refunded']).all()


상태 표준화 전: 완전중복 제거 후, FK 격리 전


,order_status,count
0,completed,175
1,cancelled,58
2,refunded,52
3,Complete,2
4,cancel,2
5,Refund,2
6,complete,1
7,COMPLETED,1
8,완료,1
9,Cancel,1


최종 clean 상태 빈도


,order_status,count
0,completed,179
1,cancelled,62
2,refunded,57


실제로 바뀐 표기


,dataset,raw_row_number,column,action,before,after
493,orders,12,order_status,status_standardization,complete,completed
494,orders,30,order_status,status_standardization,Complete,completed
495,orders,48,order_status,status_standardization,COMPLETED,completed
496,orders,64,order_status,status_standardization,완료,completed
497,orders,82,order_status,status_standardization,cancel,cancelled
498,orders,100,order_status,status_standardization,Cancel,cancelled
499,orders,118,order_status,status_standardization,CANCELLED,cancelled
500,orders,136,order_status,status_standardization,취소,cancelled
501,orders,154,order_status,status_standardization,refund,refunded
502,orders,172,order_status,status_standardization,Refund,refunded


허용값 밖 상태: 0


In [9]:
print('보존한 이상 후보: 18세 미만 또는 100세 초과')
display(customers_clean.loc[customers_clean['age_outlier_candidate'],['customer_id','age']])
print('IQR 상한:', result['manifest']['candidate_thresholds'])
display(products_clean.loc[products_clean['price_outlier_candidate'],['product_id','price']])
display(order_items_clean.loc[order_items_clean['quantity_outlier_candidate'],['order_item_id','quantity','unit_price','line_total']])
print('분리한 행: 사유 중복을 합산하면 같은 행을 두 번 셀 수 있으므로 stage별 행 수를 사용')
display(result['exclusions'].groupby(['dataset','stage']).size().reset_index(name='excluded_rows'))
display(result['exclusions'].query("stage == 'numeric_rule'"))

# 원래 과제의 describe/범위 점검: 실제 최솟값·최댓값과 남은 결측을 확인한다.
numeric_ranges = pd.DataFrame([{'dataset':name,'column':col,'count':int(df[col].count()),
    'min':df[col].min(),'max':df[col].max(),'missing':int(df[col].isna().sum()),
    'nonpositive':int(df[col].le(0).sum())}
    for name,df,col in [('customers',customers_clean,'age'),('products',products_clean,'price'),
                        ('order_items',order_items_clean,'quantity'),('order_items',order_items_clean,'unit_price')]])
display(numeric_ranges)
numeric_ranges.to_csv(REPORT_DIR/'ch05_numeric_ranges.csv',index=False,encoding='utf-8-sig')


보존한 이상 후보: 18세 미만 또는 100세 초과


,customer_id,age
32,33,150.0


IQR 상한: {'products.price_iqr_upper': 304000.0, 'order_items.quantity_iqr_upper': 7.0}


,product_id,price
94,99,9999999


,order_item_id,quantity,unit_price,line_total
698,738,50,96000,4800000
711,751,100,155000,15500000


분리한 행: 사유 중복을 합산하면 같은 행을 두 번 셀 수 있으므로 stage별 행 수를 사용


,dataset,stage,excluded_rows
0,customers,duplicate,2
1,order_items,duplicate,2
2,order_items,foreign_key,31
3,order_items,numeric_rule,8
4,orders,duplicate,2
5,orders,foreign_key,2
6,products,duplicate,2
7,products,numeric_rule,4


,dataset,raw_row_number,stage,reasons
8,products,19,numeric_rule,price_conversion_failed
9,products,49,numeric_rule,nonpositive_price_candidate
10,products,74,numeric_rule,price_conversion_failed
11,products,96,numeric_rule,nonpositive_price_candidate
12,order_items,278,numeric_rule,unit_price_conversion_failed
13,order_items,320,numeric_rule,quantity_conversion_failed
14,order_items,434,numeric_rule,unit_price_conversion_failed
15,order_items,448,numeric_rule,quantity_conversion_failed
16,order_items,512,numeric_rule,nonpositive_quantity_candidate
17,order_items,580,numeric_rule,nonpositive_unit_price_candidate


,dataset,column,count,min,max,missing,nonpositive
0,customers,age,150,18.0,150.0,0,0
1,products,price,96,5000.0,9999999.0,0,0
2,order_items,quantity,724,1.0,100.0,0,0
3,order_items,unit_price,724,5000.0,197000.0,0,0


In [10]:
# 제외 금액의 영향: 계산 가능한 원래 상세만 합계, 계산 불가 행 수는 따로 표시한다.
excluded_items = result['quarantine']['order_items'].copy()
q_text = excluded_items['quantity'].astype('string').str.strip().str.replace(',', '', regex=False)
q_text = q_text.str.replace(r'^(\d+)개$', r'\1', regex=True)
q = pd.to_numeric(q_text, errors='coerce')
p = pd.to_numeric(excluded_items['unit_price'].astype('string').str.replace(',', '', regex=False), errors='coerce')
known_amount = q * p
impact = pd.DataFrame({'scope':['격리된 원래 주문 상세(완전중복 포함)'],
    'rows':[len(excluded_items)], 'amount_calculable_rows':[int(known_amount.notna().sum())],
    'amount_unknown_rows':[int(known_amount.isna().sum())], 'known_line_total_sum':[known_amount.sum()]})
display(impact)
print('이 합은 실제 손실매출이 아니며, 원본 상세 중 분리된 계산 가능 금액이다.')


,scope,rows,amount_calculable_rows,amount_unknown_rows,known_line_total_sum
0,격리된 원래 주문 상세(완전중복 포함),41,37,4,9374000


이 합은 실제 손실매출이 아니며, 원본 상세 중 분리된 계산 가능 금액이다.


### 결과 관찰

완전중복은 파일별 2건씩 분리했다. 그 뒤 고유 ID 충돌은 0건이다. 표준화한 최종 주문 상태는 completed **179건**, cancelled **62건**, refunded **57건**이고 허용값 밖 값은 0건이다. 카테고리에서는 `전자 기기→전자기기`, `생활 용품→생활용품`, `뷰 티→뷰티`만 명시적으로 매핑했다.

상품의 숫자 규칙으로 4행, 상세의 숫자 규칙으로 8행을 일반 판매 분석에서 분리했다. 양수이지만 큰 값은 보존했다. 가격 IQR 상한은 304,000원으로 후보 상품 1개, 수량 상한은 7개로 후보 상세 2행이다. 나이 150도 후보 1행이다. 최종 숫자 범위는 age 18–150세, price 5,000–9,999,999원, quantity 1–100개, unit_price 5,000–197,000원이며 네 숫자 컬럼의 결측·비양수는 0건이다.

### 나의 해석과 판단

완전중복과 주문 상세의 주문 ID 반복을 다르게 처리했다. 0원·음수는 증정이나 반품일 수 있으므로 오류라고 확정하지 않았다. 이번에는 ‘양의 수량·금액을 갖는 일반 판매 상세’라는 범위를 정해 따로 보관했다. 큰 양수는 값의 의미가 불명확하므로 지우지 않고 플래그를 남겼다.

### 업무·분석적 의미

한 상태가 여러 표기로 나뉘면 completed 필터에서 일부 주문이 빠진다. 반대로 허용되지 않은 상태를 임의로 completed로 채우면 금액이 부풀 수 있다. 이상 후보가 포함된 결과와 제외한 결과를 다음 장에서 비교할 가치가 있다.

### 한계와 추가 확인 사항

합계에서 분리된 금액을 실제 매출 손실로 해석할 수 없다. 비양수와 큰 값의 의미는 업무 확인이 필요하고, IQR은 통계적 후보 기준일 뿐 오류 판정 기준은 아니다.


## 4. PK/FK와 파생 컬럼 재검증 — Evidence 4


In [11]:
print('원본 FK 검사')
display(result['relationship_before'])
print('최종 clean FK 검사')
display(result['relationship_after'])
assert result['relationship_after']['invalid_count'].eq(0).all()
print('FK에 따라 분리된 행과 사유')
display(result['exclusions'].query("stage == 'foreign_key'"))
assert order_items_clean['line_total'].eq(order_items_clean['quantity'] * order_items_clean['unit_price']).all()
assert orders_clean.loc[orders_clean['order_date'].notna(),'order_month'].eq(
    orders_clean.loc[orders_clean['order_date'].notna(),'order_date'].dt.to_period('M').astype('string')).all()
display(orders_clean[['order_id','order_date','order_month','order_dayofweek']].head())
display(order_items_clean[['order_item_id','quantity','unit_price','line_total']].head())
print('line_total 계산식 일치: PASS / 전체 상세금액:', f"{order_items_clean['line_total'].sum():,.0f}원")


원본 FK 검사


,relationship,invalid_count
0,orders.customer_id → customers.customer_id,2
1,order_items.order_id → orders.order_id,2
2,order_items.product_id → products.product_id,2


최종 clean FK 검사


,relationship,invalid_count
0,orders.customer_id → customers.customer_id,0
1,order_items.order_id → orders.order_id,0
2,order_items.product_id → products.product_id,0


FK에 따라 분리된 행과 사유


,dataset,raw_row_number,stage,reasons
20,orders,127,foreign_key,customer_fk_absent_in_raw_master
21,orders,279,foreign_key,customer_fk_absent_in_raw_master
22,order_items,23,foreign_key,product_fk_removed_by_cleaning
23,order_items,33,foreign_key,product_fk_removed_by_cleaning
24,order_items,78,foreign_key,product_fk_removed_by_cleaning
25,order_items,91,foreign_key,product_fk_removed_by_cleaning
26,order_items,100,foreign_key,product_fk_removed_by_cleaning
27,order_items,107,foreign_key,product_fk_removed_by_cleaning
28,order_items,131,foreign_key,product_fk_removed_by_cleaning
29,order_items,151,foreign_key,order_fk_absent_in_raw_master


,order_id,order_date,order_month,order_dayofweek
0,1,2026-07-12,2026-07,Sunday
1,2,2025-10-02,2025-10,Thursday
2,3,2026-03-13,2026-03,Friday
3,4,2025-10-12,2025-10,Sunday
4,5,2026-04-17,2026-04,Friday


,order_item_id,quantity,unit_price,line_total
0,1,2,172000,344000
1,2,4,79000,316000
2,3,3,87000,261000
3,4,1,187000,187000
4,5,5,125000,625000


line_total 계산식 일치: PASS / 전체 상세금액: 244,374,000원


In [12]:
# 상세가 일부만 남은 주문과 상세가 전부 빠진 주문을 구분한다.
raw_item_counts = raw_data['order_items'].drop_duplicates().groupby('order_id').size()
clean_item_counts = order_items_clean.groupby('order_id').size()
retention = orders_clean[['order_id','order_date','order_status']].copy()
retention['raw_detail_rows'] = retention['order_id'].map(raw_item_counts).fillna(0).astype(int)
retention['clean_detail_rows'] = retention['order_id'].map(clean_item_counts).fillna(0).astype(int)
retention['partly_retained'] = retention['clean_detail_rows'].gt(0) & retention['clean_detail_rows'].lt(retention['raw_detail_rows'])
retention['no_detail_left'] = retention['raw_detail_rows'].gt(0) & retention['clean_detail_rows'].eq(0)
eda_scope = retention['order_status'].eq('completed') & retention['order_date'].notna() & retention['order_date'].le(pd.Timestamp('2026-10-08'))
print('전체 clean 주문 중 상세 일부 보존:', int(retention['partly_retained'].sum()))
print('Chapter06 completed·유효 과거 주문 중 상세 일부 보존:', int(retention.loc[eda_scope,'partly_retained'].sum()))
print('Chapter06 범위 중 상세가 전부 빠진 주문')
display(retention.loc[eda_scope & retention['no_detail_left']])
retention.to_csv(REPORT_DIR / 'ch05_order_detail_retention.csv', index=False, encoding='utf-8-sig')


전체 clean 주문 중 상세 일부 보존: 31
Chapter06 completed·유효 과거 주문 중 상세 일부 보존: 18
Chapter06 범위 중 상세가 전부 빠진 주문


,order_id,order_date,order_status,raw_detail_rows,clean_detail_rows,partly_retained,no_detail_left
173,175,2025-08-26,completed,1,0,False,True


### 결과 관찰

최종 clean 데이터에서 없는 customer_id/order_id/product_id를 참조하는 행은 모두 0건이다. FK 때문에 주문 2행과 상세 31행을 분리했다. 원래 부모가 없던 경우와 상품·주문을 제외해 연결이 끊긴 경우를 구분해 남겼다. 상세의 `line_total`은 전 행에서 수량×단가와 일치하고, 주문 월·요일도 만들었다.

### 나의 해석과 판단

부모 행을 정리한 뒤에는 자식 행의 참조를 다시 확인해야 한다. FK를 맞추기 위해 존재하지 않는 상품·고객을 만들어 넣지 않았다. 참조할 수 없는 행을 분리하고 원본과 사유를 보관했다. `line_total`은 상세 한 행의 금액이며 전체 합계에는 취소·환불 주문도 있어 매출이나 순매출로 부르지 않는다.

### 업무·분석적 의미

상세가 일부 분리된 주문 **31건**은 남은 상세만으로 원래 주문 전체 청구액을 복원할 수 없다. Chapter06의 completed·유효 과거 주문 범위에도 이런 주문 **18건**이 포함된다. 상세가 전부 빠진 completed 주문도 있으므로 주문 수와 상세 보유 주문 수를 구분해야 한다.

### 한계와 추가 확인 사항

정상 연결만 남았다는 사실이 거래 전체를 복원했다는 뜻은 아니다. 분리된 상세의 원래 의미가 확인되면 포함 범위와 금액을 다시 계산해야 한다.


## 5. 전처리 전/후 비교 — Evidence 5


In [13]:
display(result['comparison'])
original_column_missing = pd.DataFrame([{'dataset':name,
    'before_original_column_missing':int(raw_data[name].isna().sum().sum()),
    'after_original_column_missing':int(clean_data[name][raw_data[name].columns].isna().sum().sum()),
    'after_all_column_missing':int(clean_data[name].isna().sum().sum())}
    for name in raw_data])
display(original_column_missing)
balance = pd.DataFrame([{'dataset':name, 'raw':len(raw_data[name]), 'clean':len(clean_data[name]),
    'quarantine':len(result['quarantine'][name]),
    'balanced':len(raw_data[name]) == len(clean_data[name]) + len(result['quarantine'][name])}
    for name in raw_data])
display(balance)
assert balance['balanced'].all()
print('원본 행 처리 내역 추적: PASS')


,dataset,rows_before,columns_before,missing_cells_before,full_row_duplicates_before,key_missing_before,key_duplicates_before,rows_after,columns_after,missing_cells_after,full_row_duplicates_after,key_missing_after,key_duplicates_after
0,customers,152,6,6,2,0,2,150,11,2,0,0,0
1,products,102,4,0,2,0,2,96,5,0,0,0,0
2,orders,302,5,0,2,0,2,298,10,9,0,0,0
3,order_items,765,5,0,2,0,2,724,7,0,0,0,0


,dataset,before_original_column_missing,after_original_column_missing,after_all_column_missing
0,customers,6,2,2
1,products,0,0,0
2,orders,0,3,9
3,order_items,0,0,0


,dataset,raw,clean,quarantine,balanced
0,customers,152,150,2,True
1,products,102,96,6,True
2,orders,302,298,4,True
3,order_items,765,724,41,True


원본 행 처리 내역 추적: PASS


### 결과 관찰

| 데이터 | 처리 전 | 처리 후 | 격리 행 | 변화 이유 |
| --- | ---: | ---: | ---: | --- |
| customers | 152×6 | 150×11 | 2 | 완전중복 2행, 처리 플래그 5열 추가 |
| products | 102×4 | 96×5 | 6 | 완전중복 2행+숫자 규칙 4행, 후보 플래그 1열 추가 |
| orders | 302×5 | 298×10 | 4 | 완전중복 2행+FK 2행, 월/요일·검사 플래그 5열 추가 |
| order_items | 765×5 | 724×7 | 41 | 완전중복 2행+숫자 규칙 8행+FK 31행, 금액·후보 2열 추가 |

원래 컬럼 기준 결측은 customers 6→2셀, products 0→0셀, orders 0→3셀, order_items 0→0셀이다. **orders의 전체 결측 9셀은 주문일 3셀+월 3셀+요일 3셀**로 같은 날짜 실패에서 나온다. 오류가 9건이라는 뜻은 아니다.

### 나의 해석과 판단

모든 파일에서 `원본 행 수 = clean 행 수 + quarantine 행 수`가 성립했다. 행 수 감소는 명시한 처리 기준으로 설명되며 누락된 행이 어디에 보관됐는지도 확인했다. 결측을 0으로 만드는 것보다 실패한 날짜를 숨기지 않는 편이 이번 목적에 맞다.

### 업무·분석적 의미

처리 전후 shape만 보면 이유를 알 수 없다. 격리 사유와 원래 컬럼 기준 결측을 같이 보여주면 정보 손실과 새로 발견한 오류를 구분할 수 있다.

### 한계와 추가 확인 사항

서로 다른 사유를 동시에 가진 한 행이 있으므로 사유 건수를 단순 합산하면 격리 행 수와 다를 수 있다. 위 표는 중복되지 않는 실제 격리 행 수를 사용했다.


## 6. 재현 가능한 전처리 확인 — Evidence 6

네 clean CSV와 보고서가 실제로 저장됐는지 확인한 뒤, 함수 재실행과 터미널용 스크립트 실행 전후의 해시를 비교한다. 저장 후 네 CSV를 다시 읽어 구조·값·날짜를 검증한다.

아래 표와 `ch05_reproduction_checks.json`의 온라인 PENDING은 2026-10-08 로컬 실행 당시 기록이다. 최신 제출 상태는 상단 제출 정보와 최종 체크를 따른다.


In [14]:
required = [PROCESSED_DIR / f'{name}_clean.csv' for name in raw_data]
file_checks = pd.DataFrame([{'file':p.name,'exists':p.exists(),'bytes':p.stat().st_size,'sha256':sha256(p)} for p in required])
display(file_checks)
assert len(required) == 4 and file_checks['exists'].all() and file_checks['bytes'].gt(0).all()
report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
print('요약 보고서:', report_path.relative_to(PROJECT_ROOT), 'exists=',report_path.exists(), 'bytes=',report_path.stat().st_size)
assert report_path.exists() and report_path.stat().st_size > 0
hash_before_rerun = {p.name:sha256(p) for p in required}
report_hash_before = sha256(report_path)
result_again = run_preprocessing(PROJECT_ROOT)
hash_after_rerun = {p.name:sha256(p) for p in required}
assert hash_before_rerun == hash_after_rerun and report_hash_before == sha256(report_path)
print('함수 재실행 clean4/요약 보고서 SHA-256 동일: PASS')


,file,exists,bytes,sha256
0,customers_clean.csv,True,10358,97378fa2db226653ef8da945d14e6ed50136f5f8431e5f...
1,products_clean.csv,True,4278,8791b51dadbe63b4deef6a0ebff3e6883275213dddf709...
2,orders_clean.csv,True,21439,a121e15ebc5f2599e74c2a53717a81363d49957e92e049...
3,order_items_clean.csv,True,23047,d5c7a6faf318fff67c8c86af515b3fece14e8b9fd268b8...


요약 보고서: reports/ch05_preprocessing_summary.md exists= True bytes= 8818


함수 재실행 clean4/요약 보고서 SHA-256 동일: PASS


In [15]:
script_run = subprocess.run([sys.executable, 'scripts/preprocess_data.py'], cwd=PROJECT_ROOT,
                            text=True, capture_output=True, check=True)
print(script_run.stdout)
assert script_run.returncode == 0
assert hash_before_rerun == {p.name:sha256(p) for p in required}
assert report_hash_before == sha256(report_path)
assert raw_hash_start == {p.name:sha256(p) for p in raw_paths}
print('스크립트 종료코드:',script_run.returncode, '/ 원본보존·clean4·보고서 해시 동일: PASS')


Chapter 05 전처리 완료
    dataset  rows_before  columns_before  missing_cells_before  full_row_duplicates_before  key_missing_before  key_duplicates_before  rows_after  columns_after  missing_cells_after  full_row_duplicates_after  key_missing_after  key_duplicates_after
  customers          152               6                     6                           2                   0                      2         150             11                    2                          0                  0                     0
   products          102               4                     0                           2                   0                      2          96              5                    0                          0                  0                     0
     orders          302               5                     0                           2                   0                      2         298             10                    9                          0                  0     

In [16]:
reloaded = {name:pd.read_csv(PROCESSED_DIR / f'{name}_clean.csv') for name in raw_data}
for name in reloaded:
    expected = clean_data[name]
    actual = reloaded[name]
    assert list(expected.columns) == list(actual.columns)
    for col in ['signup_date','order_date']:
        if col in actual:
            actual[col] = pd.to_datetime(actual[col],errors='coerce',format='mixed')
    pd.testing.assert_frame_equal(expected, actual, check_dtype=False)
display(quality_summary(reloaded))
display(validate_relationships(reloaded))
print('CSV 4개 재읽기: shape·컬럼·값 동일 PASS')
print('날짜 재변환 실패/보존 결측:', {'signup_date':int(reloaded['customers']['signup_date'].isna().sum()),
                                      'order_date':int(reloaded['orders']['order_date'].isna().sum())})
print('이미 보존한 NaT 2/3건이며, 저장 때문에 새로 생긴 실패는 0건이다.')
verification = {'notebook_and_script':'PASS','raw_4_preserved':'PASS','clean_4_nonempty':'PASS',
    'clean_4_hash_equal_after_rerun':'PASS','report_hash_equal_after_rerun':'PASS','reload_4_equal':'PASS',
    'pk_fk':'PASS','github_rendering':'PENDING','etl_submission':'PENDING'}
(REPORT_DIR / 'ch05_reproduction_checks.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
display(pd.DataFrame(verification.items(),columns=['check','status']))


,dataset,rows,columns,missing_cells,full_row_duplicates,key_missing,key_duplicates
0,customers,150,11,2,0,0,0
1,products,96,5,0,0,0,0
2,orders,298,10,9,0,0,0
3,order_items,724,7,0,0,0,0


,relationship,invalid_count
0,orders.customer_id → customers.customer_id,0
1,order_items.order_id → orders.order_id,0
2,order_items.product_id → products.product_id,0


CSV 4개 재읽기: shape·컬럼·값 동일 PASS
날짜 재변환 실패/보존 결측: {'signup_date': 2, 'order_date': 3}
이미 보존한 NaT 2/3건이며, 저장 때문에 새로 생긴 실패는 0건이다.


,check,status
0,notebook_and_script,PASS
1,raw_4_preserved,PASS
2,clean_4_nonempty,PASS
3,clean_4_hash_equal_after_rerun,PASS
4,report_hash_equal_after_rerun,PASS
5,reload_4_equal,PASS
6,pk_fk,PASS
7,github_rendering,PENDING
8,etl_submission,PENDING


### 나의 해석과 판단

Notebook에서 한 번 실행된 것만으로 재현 가능하다고 결론 내리지 않았다. 같은 함수를 다시 실행하고 `python scripts/preprocess_data.py`도 실제로 실행했다. 네 CSV와 요약 보고서의 SHA-256이 같고 저장 후 다시 읽은 값도 같았다. 원본 해시도 처음과 같다.

### 업무·분석적 의미

다음 장은 이 네 파일을 그대로 사용할 수 있다. 처리 코드를 바꾸면 어떤 결과가 달라지는지 해시와 처리 기록으로 다시 확인할 수 있다.

### 한계와 추가 확인 사항

같은 환경·입력·규칙에서 재현됐다는 뜻이다. 다른 pandas 버전이나 새 데이터에서 항상 같은 의미가 유지된다는 뜻은 아니다. 실제 실행한 환경은 기존 수업 프로젝트의 가상환경이며, 이번 제출 저장소 전용 가상환경을 새로 설정했다고 기록하지 않았다.


## LLM 전처리 제안과 실제 데이터 대조

**실제로 한 요청:** Codex 작업 중 검토 모델에 아래 Prompt를 전달했고 다섯 제안을 받았다. Prompt에는 구조·품질 요약을 제공했다. Codex 검토 과정에서는 공개 수업용 합성 원본 로컬 파일의 컬럼과 일부 행도 확인했다. 실제 고객 자료를 추가하거나 개인 자료를 외부에 전송하지 않았다.

**Safe Context:** customers 152×6, products 102×4, orders 302×5, order_items 765×5. 원래 결측 age4/city2. 전체 중복은 각2. 숫자·날짜가 문자열이며 별칭·관계 문제 후보가 있다.

**Prompt**

> 강사 Chapter05 전용 raw의 구조·품질 요약만 바탕으로 전처리 규칙 5개와 검증 방법을 제안해 주세요. 원본 보존, 원래 결측과 변환 실패 구분, 중복/PK/FK 구분, 이상 후보 자동 오류 단정 금지, 개인정보 행 미전송 조건을 지켜 주세요.

**실제 LLM 응답 요약**

1. 원본을 보존하고 공백·빈 문자열·뜻이 확인된 상태 별칭만 정리하며 전후 빈도와 미허용값을 남길 것.
2. 원래 결측과 변환 실패를 구분할 것. 숫자+개처럼 단위가 명확한 경우만 복구하고 나이 중앙값 대체 여부를 남길 것. 거래 날짜·금액·ID는 임의 대체하지 말 것.
3. 전체 행 중복과 PK 충돌을 구분하고, 다른 내용이 있는 동일 ID는 전부 격리할 것. 상세의 order_id 반복은 유지할 것.
4. 비양수와 큰 값은 오류 확정 대신 후보로 볼 것. 일반 판매로 범위를 정하면 제외 행·이유·금액 영향을 보관할 것.
5. 기준일을 고정하고 미래 날짜의 과거 거래 분석 포함을 제한할 것. 부모 제외 뒤 FK와 파생 금액·재읽기를 검증할 것.

추가 응답은 ‘일부 상세가 빠진 주문의 잔여 합을 원래 주문 전체 금액으로 해석하지 말라’는 내용이었다. 실제 영향 주문 수를 계산해 위 한계에 반영했다.

| 제안 | 판단 | 실제 대조와 수정 이유 |
| --- | --- | --- |
| 원본 보존·표준화 | 사용 | 원본4 해시 동일, 공백5/3/5셀, 상태179/62/57, 허용값 밖0 |
| 변환 실패 구분·중앙값 | 수정 후 사용 | 원래 age결측4와 새실패2를 나눔. 18–100세 관측값 중앙값41 적용, 플래그6개 및 대체 전후 분포 출력 |
| 중복과 PK/FK 구분 | 사용 | 파일별 완전중복2, 충돌PK0, 최종FK0. 상세order_id 반복 보존 |
| 비양수 격리·큰 값 후보 | 수정 후 사용 | 숫자규칙 상품4/상세8행 격리, 큰 양수는 IQR후보로 보존. 격리금액 계산 가능/불가를 구분 |
| 미래 날짜·관계·저장 검증 | 수정 후 사용 | 미래2/NaT3을 clean에서 보존하고 플래그 표시. 과거 거래 EDA에서 범위를 제한. 재실행/재읽기 PASS |

LLM이 규칙을 제안한 것과 실제 데이터가 그 규칙을 통과한 것은 다르다. 위 판단은 실행한 수치·변경 기록·재검증 결과에 근거했다.


## 공식 Notebook 추가 실습 6문제

### 1. 네 데이터셋의 결측 비율을 하나의 표로 합치기

처리 전과 처리 후를 함께 표시한다. 파생 컬럼이 포함된 처리 후 비율임을 구분한다.


In [17]:
combined_missing = pd.concat([pd.DataFrame({'stage':stage,'dataset':name,'column':df.columns,
    'missing_count':df.isna().sum().values,'missing_ratio_pct':(df.isna().mean()*100).round(2).values})
    for stage,data in [('raw',raw_data),('clean',clean_data)] for name,df in data.items()],ignore_index=True)
display(combined_missing)
combined_missing.to_csv(REPORT_DIR/'ch05_missing_ratios.csv',index=False,encoding='utf-8-sig')


,stage,dataset,column,missing_count,missing_ratio_pct
0,raw,customers,customer_id,0,0.00
1,raw,customers,name,0,0.00
2,raw,customers,gender,0,0.00
3,raw,customers,age,4,2.63
4,raw,customers,city,2,1.32
5,raw,customers,signup_date,0,0.00
6,raw,products,product_id,0,0.00
7,raw,products,product_name,0,0.00
8,raw,products,category,0,0.00
9,raw,products,price,0,0.00


### 2. customers_clean의 18세 미만 또는 100세 초과 확인

후보를 확인하되 실제 오류 확정과 삭제를 구분한다.


In [18]:
age_candidates = customers_clean.loc[(customers_clean['age']<18)|(customers_clean['age']>100),['customer_id','age','age_outlier_candidate']]
display(age_candidates)
print('후보 수:',len(age_candidates), '/ 보존했으며 실제 나이는 확인되지 않았다.')


,customer_id,age,age_outlier_candidate
32,33,150.0,True


후보 수: 1 / 보존했으며 실제 나이는 확인되지 않았다.


### 3. orders_clean에서 월별 주문 건수 계산

이 표는 completed만의 매출 분석이 아니라 clean 주문 전체의 상태별 건수다. NaT와 미래 날짜도 숨기지 않는다.


In [19]:
monthly_orders = orders_clean.groupby(['order_month','order_status'],dropna=False).size().unstack('order_status',fill_value=0)
monthly_orders['all_status_orders'] = monthly_orders.sum(axis=1)
display(monthly_orders)
assert int(monthly_orders['all_status_orders'].sum()) == len(orders_clean)
print('월별건수합 = clean 주문298건: PASS')
monthly_orders.to_csv(REPORT_DIR/'ch05_monthly_order_counts.csv',encoding='utf-8-sig')


order_status,cancelled,completed,refunded,all_status_orders
order_month,,,,
2025-07,6,11,2,19
2025-08,4,18,4,26
2025-09,3,4,5,12
2025-10,2,17,3,22
2025-11,6,16,1,23
2025-12,5,12,6,23
2026-01,8,8,9,25
2026-02,3,12,4,19
2026-03,6,13,3,22


월별건수합 = clean 주문298건: PASS


### 4. line_total 상위 10행

clean 상세 전체에서 계산한다. completed·유효 과거 주문만의 금액과 범위가 다르므로 매출 순위라고 이름 붙이지 않았다.


In [20]:
top10 = order_items_clean.sort_values('line_total',ascending=False).head(10)
display(top10)
top10.to_csv(REPORT_DIR/'ch05_line_total_top10.csv',index=False,encoding='utf-8-sig')


,order_item_id,order_id,product_id,quantity,unit_price,quantity_outlier_candidate,line_total
711,751,295,31,100,155000,True,15500000
698,738,290,96,50,96000,True,4800000
305,320,129,23,5,197000,False,985000
281,294,119,36,5,193000,False,965000
23,25,11,65,5,193000,False,965000
20,21,9,36,5,193000,False,965000
9,10,5,65,5,193000,False,965000
110,117,51,65,5,193000,False,965000
278,291,117,47,5,188000,False,940000
361,377,151,99,5,187000,False,935000


### 5. 전처리 전후 행 수가 줄어든 이유

customers는 중복 2행, products는 중복2+숫자규칙4행, orders는 중복2+FK2행, order_items는 중복2+숫자규칙8+FK31행이 분리됐다. 총 격리 행은 각각2/6/4/41행이며 원래 행은 quarantine에서 찾을 수 있다. raw를 덮어쓰거나 제외 행을 숨기지 않았다.

### 6. 결측 처리 기준 검토 Prompt

> 고객 150명에서 age 원래 결측4건과 숫자 변환 실패2건을 18–100세 관측 중앙값41로 채웠고 age_imputed 플래그를 남겼습니다. city 결측2건은 Unknown으로 분리했습니다. 가입일 실패2건과 주문일 실패3건은 NaT로 남겼습니다. 이 기준의 정보 손실·왜곡 가능성을 검토하고, 날짜·거래 금액·ID를 대표값으로 대체하지 않는 이유를 설명해 주세요. 원본 행을 요구하지 말고 처리 전후 검증 항목을 제안해 주세요.

위 Prompt도 Codex 검토 모델에 실제로 전달해 다음 응답을 받았다.

**실제 응답 요약:** age6건을41로 채우면 분포가 중앙에 몰릴 수 있다. 원래 결측4건과 변환 실패2건의 사유를 구분하고 미대체 관측값과 대체 후 평균·중앙값·연령대 빈도 및 대체 비율을 비교하라. 도시 Unknown2건을 실제 도시로 해석하지 말라. 날짜·금액·ID를 대표값으로 채우면 월별 거래·합계·관계가 임의로 만들어질 수 있으므로 피하라. 날짜 실패의 포함·제외 수와 금액 영향, PK/FK·원본 보존·재실행을 검증하라.

아래에서는 이 추가 응답을 실제 데이터로 대조한다.


In [21]:
# 추가 LLM 응답 검증: age 대체 사유와 연령대, 날짜 범위의 영향을 직접 계산한다.
original_customers = raw_data['customers'].drop_duplicates().copy()
original_age_numeric = pd.to_numeric(original_customers['age'],errors='coerce')
age_reason = pd.DataFrame({'customer_id':original_customers['customer_id'],
    'original_age':original_customers['age'],
    'reason':np.where(original_customers['age'].isna(),'original_missing',
                      np.where(original_age_numeric.isna(),'conversion_failed','observed'))})
display(age_reason.loc[age_reason['reason'].ne('observed')])
print('age 대체 비율:',f"{customers_clean['age_imputed'].mean()*100:.2f}%",
      '/ city Unknown 대체 비율:',f"{customers_clean['city_imputed'].mean()*100:.2f}%")
bins=[17,29,39,49,59,69,100]
labels=['18–29','30–39','40–49','50–59','60–69','70–100']
before_band=pd.cut(age_before_valid,bins=bins,labels=labels).value_counts(sort=False)
after_band=pd.cut(age_after_valid,bins=bins,labels=labels).value_counts(sort=False)
age_band=pd.DataFrame({'observed_before':before_band,'after_imputation':after_band})
age_band['difference']=age_band['after_imputation']-age_band['observed_before']
display(age_band)
age_band.to_csv(REPORT_DIR/'ch05_age_imputation_bands.csv',encoding='utf-8-sig')
# 같은 clean 상세에서 주문일 범위만 나눠 금액/행/주문 수 영향을 확인한다.
linked_items=order_items_clean.merge(orders_clean[['order_id','order_date','order_status']],
                                    on='order_id',how='left',validate='many_to_one')
date_impact=[]
for label,mask in [('날짜실패',linked_items['order_date'].isna()),
                   ('미래날짜후보',linked_items['order_date'].gt(pd.Timestamp('2026-10-08'))),
                   ('유효과거',linked_items['order_date'].notna() & linked_items['order_date'].le(pd.Timestamp('2026-10-08')))]:
    selected=linked_items.loc[mask]
    date_impact.append({'scope':label,'detail_rows':len(selected),'orders_with_detail':selected['order_id'].nunique(),
                       'all_status_line_total':selected['line_total'].sum(),
                       'completed_line_total':selected.loc[selected['order_status'].eq('completed'),'line_total'].sum()})
date_impact=pd.DataFrame(date_impact)
display(date_impact)
assert date_impact['detail_rows'].sum()==len(order_items_clean)
assert date_impact['all_status_line_total'].sum()==order_items_clean['line_total'].sum()
date_impact.to_csv(REPORT_DIR/'ch05_date_scope_impact.csv',index=False,encoding='utf-8-sig')
print('연령대 비교·날짜 범위 영향·합계 일치: PASS')


,customer_id,original_age,reason
11,12,NaN,original_missing
23,24,unknown,conversion_failed
52,53,NaN,original_missing
87,88,NaN,original_missing
96,97,unknown,conversion_failed
120,121,NaN,original_missing


age 대체 비율: 4.00% / city Unknown 대체 비율: 1.33%


,observed_before,after_imputation,difference
age,,,
18–29,35,35,0
30–39,31,31,0
40–49,24,30,6
50–59,23,23,0
60–69,29,29,0
70–100,1,1,0


,scope,detail_rows,orders_with_detail,all_status_line_total,completed_line_total
0,날짜실패,9,3,2588000,1903000
1,미래날짜후보,7,2,2623000,909000
2,유효과거,708,292,239163000,134523000


연령대 비교·날짜 범위 영향·합계 일치: PASS


### 추가 LLM 응답에 대한 판단

- **사용:** 원래 age 결측4건과 변환 실패2건을 원본 값·행별 변경 기록으로 구분했고 위에서 다시 출력했다. 대체6/150=4.00%, 도시 대체2/150=1.33%를 확인했다.
- **수정 후 사용:** 연령 분석은 나이150 후보를 분리한 관측값143개와 대체 후149개를 비교했다. 41세를 채운6명이 40–49 구간에 더해지는 것을 확인했다. 18–100은 공식 추가문제의 점검 범위이지 실제 나이의 진실을 판정하는 기준은 아니다.
- **사용:** 가입일 NaT2·주문일 NaT3을 유지했고 날짜 실패·미래·유효 과거 범위를 나눠 상세금액과 행 수를 확인했다. 날짜 실패 상세9행의 금액은 2,588,000원(그중 completed 1,903,000원), 미래 후보 상세7행은 2,623,000원(그중 completed 909,000원)이다. 유효 과거 completed 상세금액은 134,523,000원이다. 같은 분할의 합은 clean 전체와 일치한다. PK/FK와 원본 보존·재실행 검증도 PASS이다.
- **보류한 작업:** 실제 나이·날짜·금액·ID를 추정해 복원하는 작업은 근거가 없어 수행하지 않았다. LLM의 검증 제안과 원래 값을 복원했다는 주장을 구분한다.

가장 큰 한계는 대체값이 실제 나이가 아니고, 제외된 상세의 거래 의미가 아직 확인되지 않았다는 점이다. 이 추가 검토는 결측을 단순히 없애기보다 그 영향을 드러내는 방향으로 반영했다.


## 7. Chapter 05 최종 판단

### 내가 정한 전처리 원칙 3가지

1. 원본을 보존하고 처리된 파일·격리된 행·사유를 구분해 남긴다.
2. 변환 실패와 이상 후보를 숨기지 않고, 의미가 확인된 표기만 바꾼다.
3. 행을 바꾼 뒤 PK/FK·파생식·저장 후 값·재실행 결과를 다시 검증한다.

### 가장 위험하다고 생각한 자동 처리

‘결측과 이상값을 모두 지우면 데이터가 깨끗해진다’는 처리다. 날짜 실패나 비양수 상세를 일괄 삭제하고 영향 기록을 남기지 않으면, 표가 그럴듯해도 어떤 주문이 빠졌는지 알 수 없다. 고유 ID 충돌에서 첫 행만 남기는 방식도 같은 이유로 위험하다.

### 다음 EDA에서 주의할 데이터 특성

상태는 표준화됐지만 취소·환불이 포함돼 있으므로 금액 분석 범위를 먼저 정한다. 날짜 분석에서는 유효 주문일≤2026-10-08의 completed 주문을 일관되게 사용한다. 미래 날짜 2건·NaT3건, 나이 대체6건·후보1건, 가격후보1개·수량후보2행과 일부 상세만 남은 주문을 구분한다.

### 현재 전처리의 한계

실제 업무 의미를 확인한 것은 아니므로 격리값이 오류라는 결론은 내리지 않는다. 일부 상세가 빠진 주문의 원래 청구액과 날짜 실패의 월을 복원할 수 없다. 나이 대체와 큰 수량이 분포·합계에 주는 영향을 다음 장에서 검토할 필요가 있다. 합성 샘플 결과를 실제 고객 행동이나 인과관계로 확대하지 않는다.

## Chapter 05 완료 Evidence

| 항목 | 실제 상태 |
| --- | --- |
| Notebook 실행 및 저장된 출력 | PASS |
| 원본 CSV4 해시 보존 | PASS |
| 문자열·숫자·날짜 실패 점검 | PASS |
| 결측·중복·이상 후보·처리 사유 기록 | PASS |
| PK/FK·파생 컬럼 | PASS |
| processed CSV4 및 요약 보고서 | PASS |
| 함수·스크립트 재실행과 해시 비교 | PASS |
| 저장 CSV4 재읽기 | PASS |
| LLM 제안과 실제 수치 대조 | PASS |
| GitHub 게시·최종 렌더링 | PENDING |
| eTL 최종 URL 제출 | PENDING |

## 최종 제출 체크

- [x] raw 원본 네 파일을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교와 행별 격리 사유를 남겼습니다.
- [x] 처리 기준·정보 손실 가능성을 설명했습니다.
- [x] clean CSV 네 파일과 보고서, 재실행·재읽기 결과를 확인했습니다.
- [x] 수업용 합성 데이터만 사용했고 API Key·Secret·실제 고객 자료를 넣지 않았습니다.
- [ ] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시되는지 최종 확인합니다.
- [ ] 최종 Notebook 파일 URL을 eTL에 제출합니다.

위 PENDING과 빈 체크는 로컬 실행 결과와 구분한 외부 확인 상태이다. 실제 성공 확인 전에 완료로 바꾸지 않는다.
